# Distance Threshold Exceedance Analysis

## Proportion of SALs Beyond Pharmacy Access Thresholds (k = 1, 2, 3)

**Tess Vu**

Binary accessibility screening: for each SAL, does a pharmacy exist
within a given network distance threshold? This complements the 2SFCA by
providing a policy-legible metric that avoids decay functions,
demand-weighting, and provincial quantiles. The question is simple:
**can people physically reach medication or not?**

Three levels of k are analyzed separately:

- **k = 1 (reachability):** Can a person reach *any* pharmacy within the
  threshold? This is the minimum viability question.
- **k = 2 (redundancy):** Is there a second pharmacy within reach? If
  the nearest pharmacy is closed, stocked out, or overcrowded, is there
  a fallback?
- **k = 3 (choice):** Do residents have meaningful options? Three
  reachable pharmacies suggests a functioning local market rather than a
  single point of dependency.

Thresholds are applied to pre-computed k-nearest-pharmacy distances
(Euclidean, pedestrian network, drive network) from SAL centroids,
produced by the `sal_pharmacy_distance_k3` notebook.

## SETUP AND IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [ ]:
# Register Space Mono fonts.
for f in [
    "data/Space_Mono/SpaceMono-Regular.ttf",
    "data/Space_Mono/SpaceMono-Bold.ttf",
    "data/Space_Mono/SpaceMono-Italic.ttf",
    "data/Space_Mono/SpaceMono-BoldItalic.ttf",
]:
    fm.fontManager.addfont(f)

# Register Space Grotesk fonts.
for f in [
    "data/Space_Grotesk/static/SpaceGrotesk-Regular.ttf",
    "data/Space_Grotesk/static/SpaceGrotesk-Medium.ttf",
    "data/Space_Grotesk/static/SpaceGrotesk-Bold.ttf",
    "data/Space_Grotesk/static/SpaceGrotesk-Light.ttf",
    "data/Space_Grotesk/static/SpaceGrotesk-SemiBold.ttf",
]:
    fm.fontManager.addfont(f)

FONT_TITLE = "Space Mono"
FONT_BODY = "Space Grotesk"

plt.rcParams.update({
    "font.family": FONT_BODY,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.title_fontsize": 11,
})

## CONFIGURATION

All file paths and threshold definitions in a single cell. Thresholds
are set once and used throughout the notebook.

In [ ]:
# FILE PATHS
# Pre-computed SAL-to-pharmacy distances with k=1, k=2, k=3 columns.
DISTANCE_CSV = "data/networks/sal_pharmacy_distances_k3.csv"

# Population estimates with settlement type classifications.
POP_CSV = "data/pop_pred_final.csv"

# IMAGE OUTPUT
IMAGE_DIR = "images/threshold"
import os
os.makedirs(IMAGE_DIR, exist_ok = True)

# K LEVELS TO ANALYZE
K_LEVELS = [1, 2, 3]

# DISTANCE THRESHOLDS (kilometers)
# Walking thresholds reflect pedestrian-realistic access bands.
WALK_THRESHOLDS_KM = [1, 2, 3, 5]

# Driving thresholds reflect vehicle-based access bands.
DRIVE_THRESHOLDS_KM = [5, 10, 15, 20]

# Euclidean thresholds for straight-line baseline comparison.
EUCLIDEAN_THRESHOLDS_KM = [1, 3, 5, 10]

# PROVINCE LOOKUP from EA_CODE first digit.
# Stats SA geographic coding: 5 = KwaZulu-Natal, 7 = Gauteng.
PROVINCE_MAP = {
    "5": "KwaZulu-Natal",
    "7": "Gauteng"
}

# Settlement type display order.
SETTLEMENT_ORDER = ["Urban", "Traditional", "Farms"]

## LOAD DATA

In [ ]:
# Load pre-computed SAL-to-pharmacy distances (k = 3 version).
dist_raw = pd.read_csv(DISTANCE_CSV)
print(f"DISTANCE DATA LOADED: {dist_raw.shape[0]} rows, {dist_raw.shape[1]} columns")
print(f"Columns: {list(dist_raw.columns)}")
print()

# Load population and settlement type data.
pop_raw = pd.read_csv(POP_CSV)
print(f"POPULATION DATA LOADED: {pop_raw.shape[0]} rows, {pop_raw.shape[1]} columns")
print(f"Columns: {list(pop_raw.columns)}")

## DEDUPLICATE SALs

The SAL shapefile used in the distance notebook contains 39,177 features,
but `pop_pred_final.csv` has 38,380 unique EA_CODEs. The difference of
797 arises because `sal_w_ward_new.shp` is a SAL-to-ward spatial join
where SALs straddling ward boundaries appear once per overlapping ward
(evidenced by the `AREA` and `PERCENTAGE` columns in the shapefile).

When the distance notebook left-joins from the shapefile to population
data, each duplicate EA_CODE receives the same population estimate and
the same centroid-based distances. If not deduplicated, these 797 SALs
are **double-counted** in both SAL tallies and population-weighted
proportions.

**Fix:** Deduplicate on EA_CODE before threshold analysis, keeping the
first occurrence. Since distances are computed from the SAL centroid
(identical for duplicate entries of the same EA_CODE), no distance
information is lost.

In [ ]:
# Check for duplicate EA_CODEs in distance data.
dist_raw["EA_CODE"] = pd.to_numeric(dist_raw["EA_CODE"], errors = "coerce").astype("Int64")
n_before = len(dist_raw)
n_unique = dist_raw["EA_CODE"].nunique()
n_dupes = n_before - n_unique

print(f"DISTANCE DATA: {n_before} rows, {n_unique} unique EA_CODEs")
print(f"DUPLICATE EA_CODEs: {n_dupes}")

if n_dupes > 0:
    # Show a sample of duplicates for verification.
    dupe_mask = dist_raw.duplicated(subset = ["EA_CODE"], keep = False)
    sample_ea = dist_raw.loc[dupe_mask, "EA_CODE"].unique()[:5]
    print(f"Sample duplicate EA_CODEs: {list(sample_ea)}")
    print()

    # Deduplicate: keep first occurrence per EA_CODE.
    dist_raw = dist_raw.drop_duplicates(subset = ["EA_CODE"], keep = "first").copy()
    print(f"AFTER DEDUPLICATION: {len(dist_raw)} rows")
else:
    print("No duplicates found.")

## MERGE AND PREPARE ANALYSIS TABLE

Join distances with population attributes (settlement type, economic
status, estimated population). Province is derived from the EA_CODE
prefix as a fallback if not present in the distance file.

In [ ]:
# Normalize EA_CODE to int64 on both sides.
pop_raw["EA_CODE"] = pd.to_numeric(pop_raw["EA_CODE"], errors = "coerce").astype("Int64")

# Select relevant columns from population data.
pop_cols = ["EA_CODE", "EA_GTYPE", "EA_TYPE", "econ_status", "sal2023_est", "area_km2"]
pop_subset = pop_raw[pop_cols].copy()

# Drop sal2023_est from distance data if it exists (prefer population version).
dist_raw = dist_raw.drop(columns = ["sal2023_est"], errors = "ignore")

# Merge distance data with population attributes.
df = dist_raw.merge(pop_subset, on = "EA_CODE", how = "left")

print(f"MERGED SHAPE: {df.shape}")
print(f"Match rate: {df['EA_GTYPE'].notna().sum()} / {len(df)} ({df['EA_GTYPE'].notna().mean() * 100:.1f}%)")

In [ ]:
# Derive province from EA_CODE if not already present.
if "PR_NAME" not in df.columns:
    df["PR_NAME"] = df["EA_CODE"].astype(str).str[0].map(PROVINCE_MAP)
    print("PROVINCE DERIVED FROM EA_CODE PREFIX")
else:
    print("PROVINCE COLUMN ALREADY PRESENT IN DISTANCE DATA")

print(f"Province distribution:")
print(df["PR_NAME"].value_counts())
print()

# Confirm settlement type distribution.
print("SETTLEMENT TYPE (EA_GTYPE) DISTRIBUTION:")
print(df["EA_GTYPE"].value_counts())
print()

# Drop rows with missing province or settlement type.
before = len(df)
df = df.dropna(subset = ["PR_NAME", "EA_GTYPE"]).copy()
after = len(df)
print(f"ROWS DROPPED (missing province or settlement type): {before - after}")
print(f"ANALYSIS TABLE: {after} SALs")

In [ ]:
# Identify available distance columns for each k level.
# Column naming convention: {mode}_dist_k{ki}_km.
dist_col_map = {}

for ki in K_LEVELS:
    mode_cols = {}
    for mode, col_pattern in [("walk", f"walk_dist_k{ki}_km"),
                               ("drive", f"drive_dist_k{ki}_km"),
                               ("euclidean", f"euclidean_dist_k{ki}_km")]:
        if col_pattern in df.columns:
            mode_cols[mode] = col_pattern
            valid = df[col_pattern].notna().sum()
            print(f"  k={ki} {mode}: column '{col_pattern}' found, {valid} valid values ({valid / len(df) * 100:.1f}%)")
        else:
            print(f"  k={ki} {mode}: column '{col_pattern}' NOT FOUND")
    dist_col_map[ki] = mode_cols

print()
print(f"K LEVELS WITH DATA: {list(dist_col_map.keys())}")

## DISTANCE OVERVIEW BY PROVINCE AND SETTLEMENT TYPE

Quick summary statistics before threshold analysis. Reported for k=1
(nearest pharmacy) to establish baseline, with k=2 and k=3 medians
for comparison.

In [ ]:
# Summary statistics for k=1 distances by province.
k1_modes = dist_col_map.get(1, {})

for mode, col in k1_modes.items():
    print(f"DISTANCE SUMMARY (k=1): {mode.upper()} (km)")
    summary = df.groupby("PR_NAME")[col].describe(percentiles = [0.25, 0.5, 0.75, 0.9, 0.95])
    print(summary.round(2).to_string())
    print()

In [ ]:
# Median distance comparison across k levels by province and settlement type.
print("MEDIAN DISTANCE (km) ACROSS K LEVELS BY PROVINCE x SETTLEMENT TYPE")
print()

for mode in ["walk", "drive"]:
    print(f"  {mode.upper()} NETWORK")
    rows = []
    for ki in K_LEVELS:
        col = dist_col_map.get(ki, {}).get(mode)
        if col is None:
            continue
        pivot = df.pivot_table(
            values = col,
            index = "EA_GTYPE",
            columns = "PR_NAME",
            aggfunc = "median"
        )
        pivot = pivot.reindex(SETTLEMENT_ORDER)
        for gtype in SETTLEMENT_ORDER:
            for province in ["Gauteng", "KwaZulu-Natal"]:
                val = pivot.loc[gtype, province] if gtype in pivot.index and province in pivot.columns else np.nan
                rows.append({
                    "Settlement": gtype,
                    "Province": province,
                    "k": ki,
                    "Median (km)": round(val, 2) if pd.notna(val) else np.nan
                })

    comparison = pd.DataFrame(rows)
    comparison_pivot = comparison.pivot_table(
        values = "Median (km)",
        index = ["Province", "Settlement"],
        columns = "k"
    )
    print(comparison_pivot.round(2).to_string())
    print()

## THRESHOLD EXCEEDANCE FLAGS

For each SAL, each threshold, and each k level, flag whether the k-th
nearest pharmacy distance exceeds the threshold. A value of `True` means
the SAL **fails** the accessibility test at that threshold for that k.

For k=1 a flag of `True` means no pharmacy is within reach. For k=2 or
k=3 a flag of `True` means the SAL does not have two or three pharmacies
within reach.

In [ ]:
# Map thresholds to distance modes.
threshold_config = {
    "walk": WALK_THRESHOLDS_KM,
    "drive": DRIVE_THRESHOLDS_KM,
    "euclidean": EUCLIDEAN_THRESHOLDS_KM
}

# Create binary exceedance flags for each k level.
flag_columns = {}

for ki in K_LEVELS:
    ki_flags = []
    modes = dist_col_map.get(ki, {})

    for mode, col in modes.items():
        thresholds = threshold_config.get(mode, [])
        for t in thresholds:
            flag_col = f"exceeds_{mode}_k{ki}_{t}km"
            df[flag_col] = df[col] > t
            ki_flags.append(flag_col)

    flag_columns[ki] = ki_flags
    print(f"K={ki}: {len(ki_flags)} FLAGS CREATED")

print()

# Print overall exceedance summary for k=1.
print("K=1 EXCEEDANCE SUMMARY (all SALs)")
for fc in flag_columns.get(1, []):
    n_exceed = df[fc].sum()
    print(f"  {fc}: {n_exceed} SALs ({n_exceed / len(df) * 100:.1f}%)")

## EXCEEDANCE PROPORTIONS BY PROVINCE (k = 1, 2, 3)

Proportion of SALs exceeding each threshold, split by province. Both
unweighted (SAL count) and population-weighted proportions are computed.

The population-weighted version answers: what share of *people* live
beyond the threshold for their k-th nearest pharmacy?

In [ ]:
def compute_exceedance_table(data, flag_cols, group_col, weight_col = None):
    """Compute proportion exceeding threshold, grouped by a column.

    Parameters
    data : DataFrame with flag columns and grouping column.
    flag_cols : list of boolean flag column names.
    group_col : column to group by.
    weight_col : if provided, compute population-weighted proportions.

    Returns
    DataFrame with groups as rows and thresholds as columns.
    """
    results = {}
    for grp, grp_df in data.groupby(group_col):
        row = {}
        for fc in flag_cols:
            if weight_col and weight_col in grp_df.columns:
                total_pop = grp_df[weight_col].sum()
                exceed_pop = grp_df.loc[grp_df[fc], weight_col].sum()
                row[fc] = exceed_pop / total_pop if total_pop > 0 else np.nan
            else:
                row[fc] = grp_df[fc].mean()
        results[grp] = row
    return pd.DataFrame(results).T

In [ ]:
# UNWEIGHTED EXCEEDANCE by province for each k level.
for ki in K_LEVELS:
    print(f"UNWEIGHTED EXCEEDANCE PROPORTIONS BY PROVINCE (k = {ki}, % of SALs)")
    print()

    ki_modes = dist_col_map.get(ki, {})
    ki_flags = flag_columns.get(ki, [])

    for mode in ["walk", "drive", "euclidean"]:
        mode_flags = [f for f in ki_flags if f.startswith(f"exceeds_{mode}_k{ki}_")]
        if not mode_flags:
            continue
        table = compute_exceedance_table(df, mode_flags, "PR_NAME")
        display_cols = {f: f.replace(f"exceeds_{mode}_k{ki}_", "").replace("km", " km") for f in mode_flags}
        table = table.rename(columns = display_cols)
        print(f"  {mode.upper()} NETWORK")
        print((table * 100).round(1).to_string())
        print()
    print()

In [ ]:
# POPULATION-WEIGHTED EXCEEDANCE by province for each k level.
for ki in K_LEVELS:
    print(f"POPULATION-WEIGHTED EXCEEDANCE BY PROVINCE (k = {ki}, % of people)")
    print()

    ki_flags = flag_columns.get(ki, [])

    for mode in ["walk", "drive", "euclidean"]:
        mode_flags = [f for f in ki_flags if f.startswith(f"exceeds_{mode}_k{ki}_")]
        if not mode_flags:
            continue
        table = compute_exceedance_table(df, mode_flags, "PR_NAME", weight_col = "sal2023_est")
        display_cols = {f: f.replace(f"exceeds_{mode}_k{ki}_", "").replace("km", " km") for f in mode_flags}
        table = table.rename(columns = display_cols)
        print(f"  {mode.upper()} NETWORK")
        print((table * 100).round(1).to_string())
        print()
    print()

## EXCEEDANCE BY PROVINCE AND SETTLEMENT TYPE

The critical cross-tabulation: how do threshold exceedance rates vary
across settlement types within each province? Traditional and farm
settlements in KZN are expected to show the highest exceedance rates,
reflecting inherited apartheid-era infrastructure deficits.

Reported for k=1 (reachability) and k=3 (choice) side by side.

In [ ]:
# Cross-tabulation: province x settlement type, for k=1 and k=3.
for ki in [1, 3]:
    print(f"EXCEEDANCE PROPORTIONS BY PROVINCE x SETTLEMENT TYPE (k = {ki}, % of SALs)")
    print()

    ki_flags = flag_columns.get(ki, [])

    for mode in ["walk", "drive"]:
        mode_flags = [f for f in ki_flags if f.startswith(f"exceeds_{mode}_k{ki}_")]
        if not mode_flags:
            continue

        print(f"  {mode.upper()} NETWORK")

        for province in ["Gauteng", "KwaZulu-Natal"]:
            prov_df = df[df["PR_NAME"] == province]
            table = compute_exceedance_table(prov_df, mode_flags, "EA_GTYPE")
            table = table.reindex(SETTLEMENT_ORDER).dropna(how = "all")
            display_cols = {f: f.replace(f"exceeds_{mode}_k{ki}_", "").replace("km", " km") for f in mode_flags}
            table = table.rename(columns = display_cols)
            print(f"    {province}")
            print(f"    n = {len(prov_df)}")
            print((table * 100).round(1).to_string())
            print()
    print()

## REDUNDANCY GAP: k=1 vs k=3

This section directly compares the k=1 and k=3 exceedance rates at the
same threshold. The difference reveals how many SALs pass the basic
reachability test (k=1) but fail the choice test (k=3). These are SALs
with fragile access, dependent on a single pharmacy.

In [ ]:
# Compute the redundancy gap for walk and drive at key thresholds.
print("REDUNDANCY GAP: % SALs EXCEEDING THRESHOLD AT k=3 MINUS k=1")
print("(Positive values = SALs with access to 1 pharmacy but not 3)")
print()

key_thresholds = {
    "walk": [3, 5],
    "drive": [10, 20]
}

for mode, thresholds in key_thresholds.items():
    print(f"  {mode.upper()} NETWORK")
    for t in thresholds:
        k1_flag = f"exceeds_{mode}_k1_{t}km"
        k3_flag = f"exceeds_{mode}_k3_{t}km"

        if k1_flag not in df.columns or k3_flag not in df.columns:
            print(f"    {t} km: columns not available")
            continue

        for province in ["Gauteng", "KwaZulu-Natal"]:
            prov_df = df[df["PR_NAME"] == province]
            n = len(prov_df)
            k1_pct = prov_df[k1_flag].mean() * 100
            k3_pct = prov_df[k3_flag].mean() * 100
            gap = k3_pct - k1_pct
            print(f"    {province} @ {t} km: k1 = {k1_pct:.1f}%, k3 = {k3_pct:.1f}%, gap = {gap:.1f} pp")

    print()

## VISUALIZATIONS

Grouped bar charts comparing k=1 vs k=3 exceedance rates side by side,
highlighting the redundancy gap.

In [ ]:
# Grouped bar chart: Walk exceedance by province, k=1 vs k=3.
fig, axes = plt.subplots(1, 2, figsize = (14, 5), sharey = True)

for ax_idx, province in enumerate(["Gauteng", "KwaZulu-Natal"]):
    prov_df = df[df["PR_NAME"] == province]
    n_prov = len(prov_df)

    k_vals = [1, 3]
    x = np.arange(len(WALK_THRESHOLDS_KM))
    width = 0.35
    colors = ["#2e3e6c", "#ffcb05"]

    for j, ki in enumerate(k_vals):
        pcts = []
        for t in WALK_THRESHOLDS_KM:
            flag = f"exceeds_walk_k{ki}_{t}km"
            if flag in prov_df.columns:
                pcts.append(prov_df[flag].mean() * 100)
            else:
                pcts.append(0)

        offset = (j - 0.5) * width
        bars = axes[ax_idx].bar(x + offset, pcts, width, label = f"k = {ki}", color = colors[j])

        # Add value labels.
        for bar, pct in zip(bars, pcts):
            if pct > 3:
                axes[ax_idx].text(
                    bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                    f"{pct:.0f}%", ha = "center", va = "bottom", fontsize = 8
                )

    axes[ax_idx].set_xlabel("Walk Threshold (km)")
    axes[ax_idx].set_xticks(x)
    axes[ax_idx].set_xticklabels([f"{t}" for t in WALK_THRESHOLDS_KM])
    axes[ax_idx].set_title(f"{province} (n = {n_prov:,})", fontfamily = FONT_TITLE, fontsize = 12)
    axes[ax_idx].legend()
    axes[ax_idx].set_ylim(0, 105)

axes[0].set_ylabel("% SALs Exceeding Threshold")
fig.suptitle("Walk Network: k=1 (Reachability) vs k=3 (Choice)",
             fontfamily = FONT_TITLE, fontsize = 14, fontweight = "bold")
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, "walk_k1_vs_k3_bars.png"), dpi = 150, bbox_inches = "tight")
plt.show()
print(f"SAVED: {os.path.join(IMAGE_DIR, 'walk_k1_vs_k3_bars.png')}")

In [ ]:
# Grouped bar chart: Drive exceedance by province, k=1 vs k=3.
fig, axes = plt.subplots(1, 2, figsize = (14, 5), sharey = True)

for ax_idx, province in enumerate(["Gauteng", "KwaZulu-Natal"]):
    prov_df = df[df["PR_NAME"] == province]
    n_prov = len(prov_df)

    k_vals = [1, 3]
    x = np.arange(len(DRIVE_THRESHOLDS_KM))
    width = 0.35
    colors = ["#2e3e6c", "#ffcb05"]

    for j, ki in enumerate(k_vals):
        pcts = []
        for t in DRIVE_THRESHOLDS_KM:
            flag = f"exceeds_drive_k{ki}_{t}km"
            if flag in prov_df.columns:
                pcts.append(prov_df[flag].mean() * 100)
            else:
                pcts.append(0)

        offset = (j - 0.5) * width
        bars = axes[ax_idx].bar(x + offset, pcts, width, label = f"k = {ki}", color = colors[j])

        for bar, pct in zip(bars, pcts):
            if pct > 3:
                axes[ax_idx].text(
                    bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                    f"{pct:.0f}%", ha = "center", va = "bottom", fontsize = 8
                )

    axes[ax_idx].set_xlabel("Drive Threshold (km)")
    axes[ax_idx].set_xticks(x)
    axes[ax_idx].set_xticklabels([f"{t}" for t in DRIVE_THRESHOLDS_KM])
    axes[ax_idx].set_title(f"{province} (n = {n_prov:,})", fontfamily = FONT_TITLE, fontsize = 12)
    axes[ax_idx].legend()
    axes[ax_idx].set_ylim(0, 105)

axes[0].set_ylabel("% SALs Exceeding Threshold")
fig.suptitle("Drive Network: k=1 (Reachability) vs k=3 (Choice)",
             fontfamily = FONT_TITLE, fontsize = 14, fontweight = "bold")
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, "drive_k1_vs_k3_bars.png"), dpi = 150, bbox_inches = "tight")
plt.show()
print(f"SAVED: {os.path.join(IMAGE_DIR, 'drive_k1_vs_k3_bars.png')}")

## HEATMAPS: SETTLEMENT TYPE x THRESHOLD

Heatmaps for walk and drive networks showing exceedance rate by
province-settlement type combination. Separate panels for k=1 and k=3.

In [ ]:
# Build heatmap data for walk and drive, k=1 and k=3.
for mode, thresholds in [("walk", WALK_THRESHOLDS_KM), ("drive", DRIVE_THRESHOLDS_KM)]:
    fig, axes = plt.subplots(1, 2, figsize = (16, 5), sharey = True)

    for ax_idx, ki in enumerate([1, 3]):
        ki_flags = flag_columns.get(ki, [])
        mode_flags = [f for f in ki_flags if f.startswith(f"exceeds_{mode}_k{ki}_")]
        if not mode_flags:
            continue

        # Build matrix: rows = province/settlement, columns = thresholds.
        heatmap_rows = []
        row_labels = []

        for province in ["Gauteng", "KwaZulu-Natal"]:
            prov_df = df[df["PR_NAME"] == province]
            for gtype in SETTLEMENT_ORDER:
                subset = prov_df[prov_df["EA_GTYPE"] == gtype]
                if len(subset) == 0:
                    continue
                prov_short = province[:3]
                row_labels.append(f"{prov_short} / {gtype}")
                row_vals = []
                for fc in mode_flags:
                    row_vals.append(subset[fc].mean() * 100)
                heatmap_rows.append(row_vals)

        matrix = np.array(heatmap_rows)
        col_labels = [f"{t} km" for t in thresholds]

        im = axes[ax_idx].imshow(matrix, cmap = "YlOrRd", aspect = "auto", vmin = 0, vmax = 100)

        axes[ax_idx].set_xticks(range(len(col_labels)))
        axes[ax_idx].set_xticklabels(col_labels)
        axes[ax_idx].set_yticks(range(len(row_labels)))
        axes[ax_idx].set_yticklabels(row_labels)

        # Annotate cells.
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                val = matrix[i, j]
                color = "white" if val > 60 else "black"
                axes[ax_idx].text(j, i, f"{val:.0f}%", ha = "center", va = "center",
                                  color = color, fontsize = 9)

        axes[ax_idx].set_title(f"k = {ki}", fontfamily = FONT_TITLE, fontsize = 12)

    fig.suptitle(f"{mode.capitalize()} Network: % SALs Exceeding Threshold",
                 fontfamily = FONT_TITLE, fontsize = 14, fontweight = "bold")
    fig.colorbar(im, ax = axes, label = "% Exceeding", shrink = 0.8)
    plt.tight_layout()

    save_path = os.path.join(IMAGE_DIR, f"heatmap_{mode}_k1_k3.png")
    plt.savefig(save_path, dpi = 150, bbox_inches = "tight")
    plt.show()
    print(f"SAVED: {save_path}")

## POLICY SUMMARY TABLE

Formatted output for direct use in reports. Each row answers: at
threshold X, what proportion of SALs (and people) in each province
cannot reach their k-th nearest pharmacy?

In [ ]:
# Build a clean summary table for report use.
summary_rows = []

for ki in K_LEVELS:
    ki_modes = dist_col_map.get(ki, {})

    for mode in ["walk", "drive"]:
        col = ki_modes.get(mode)
        if col is None:
            continue
        thresholds = threshold_config[mode]

        for t in thresholds:
            flag_col = f"exceeds_{mode}_k{ki}_{t}km"
            if flag_col not in df.columns:
                continue

            for province in ["Gauteng", "KwaZulu-Natal"]:
                prov_df = df[df["PR_NAME"] == province]
                n_total = len(prov_df)
                n_exceed = prov_df[flag_col].sum()
                pct_sal = n_exceed / n_total * 100

                pop_total = prov_df["sal2023_est"].sum()
                pop_exceed = prov_df.loc[prov_df[flag_col], "sal2023_est"].sum()
                pct_pop = pop_exceed / pop_total * 100 if pop_total > 0 else np.nan

                summary_rows.append({
                    "k": ki,
                    "Mode": mode.capitalize(),
                    "Threshold (km)": t,
                    "Province": province,
                    "SALs Total": n_total,
                    "SALs Exceeding": int(n_exceed),
                    "% SALs": round(pct_sal, 1),
                    "Pop Exceeding": int(pop_exceed),
                    "% Pop": round(pct_pop, 1)
                })

summary_df = pd.DataFrame(summary_rows)
print("POLICY SUMMARY TABLE")
print(summary_df.to_string(index = False))

## EXPORT RESULTS

In [ ]:
# Export the analysis table with all threshold flags.
export_cols = ["EA_CODE", "PR_NAME", "EA_GTYPE", "EA_TYPE", "econ_status", "sal2023_est"]

# Add distance columns for all k levels.
for ki in K_LEVELS:
    for mode, col in dist_col_map.get(ki, {}).items():
        export_cols.append(col)

# Add all flag columns.
for ki in K_LEVELS:
    export_cols.extend(flag_columns.get(ki, []))

# Filter to columns that exist.
export_cols = [c for c in export_cols if c in df.columns]

# Export full analysis table.
full_export_path = "data/networks/sal_threshold_flags_k3.csv"
df[export_cols].to_csv(full_export_path, index = False)
print(f"EXPORTED: {full_export_path}")
print(f"Rows: {len(df)}, Columns: {len(export_cols)}")
print()

# Export the summary table.
summary_export_path = "data/networks/threshold_summary_k3.csv"
summary_df.to_csv(summary_export_path, index = False)
print(f"EXPORTED: {summary_export_path}")

## NOTES AND LIMITATIONS

- **Deduplication applied.** The SAL shapefile (`sal_w_ward_new.shp`)
  contains \~797 SALs that appear multiple times due to SAL-to-ward
  spatial overlay. This notebook deduplicates on EA_CODE before analysis
  to avoid inflating SAL counts and population totals. The prior version
  of this notebook did not deduplicate, which slightly inflated all
  reported proportions.

- **k=1 vs k=3 interpretation.** k=1 exceedance answers "is there any
  pharmacy within reach?" while k=3 exceedance answers "are there three
  pharmacies within reach?" A SAL that passes k=1 but fails k=3 has
  fragile access, dependent on a single facility. The gap between k=1
  and k=3 exceedance rates is the "redundancy deficit."

- **Centroid assumption.** Distances are measured from geometric SAL
  centroids. For large or irregularly shaped SALs (common in rural KZN),
  some residents may be significantly farther from the nearest pharmacy
  than the centroid distance suggests.

- **OSM network completeness.** Missing roads in informal settlements
  and rural areas cause the algorithm to route around gaps, inflating
  computed distances. Walk distances in these areas may overestimate
  true pedestrian travel where informal footpaths exist.

- **Threshold selection.** The thresholds (e.g., 3 km walk, 10 km drive)
  are policy-relevant reference points but inherently arbitrary. A SAL
  at 3.1 km is classified differently from one at 2.9 km despite
  near-identical access. The multi-threshold approach mitigates this by
  showing the full exceedance curve rather than a single binary cut.

- **No transit mode.** The dominant transport mode for many South
  Africans (minibus taxi) is not captured in either the walk or drive
  network. A SAL classified as exceeding the walk threshold may be
  well-served by a taxi route that passes through a pharmacy cluster.

- **Pharmacy universe.** The k=3 distance file uses
  `PHARMACIES_MASTER_FINAL.csv` (2,241 pharmacies). The older k=1
  notebook (`osmnx_network_download`) used
  `PHARMACIES_places_full_results.csv` (4,539 pharmacies). The
  threshold results reported here are based on the 2,241-pharmacy
  universe, which is the deduplicated and quality-checked set.